# 03 — Boltz-2 Predictions (Screening)

This notebook prepares and submits Boltz-2 structure prediction jobs for LigandMPNN designs.
It is optimized for batch processing using Slurm Job Arrays.

**Key Features:**
- Generates Boltz YAML inputs (No MSA, De Novo mode)
- Configures "Fast Screen" settings (No Affinity optimization initially)
- Creates and submits large Slurm Arrays for high-throughput prediction

In [5]:
import os
import re
import json
import yaml
import shutil
import datetime
import math
from pathlib import Path
import pandas as pd
from Bio import SeqIO
from tqdm.auto import tqdm
from IPython.display import display

# =============================================================================
# 1) CONFIGURATION
# =============================================================================
PROJECT_ROOT = Path("/private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026")

# LigandMPNN outputs (Same source as RF3)
MPNN_BASE_DIR = PROJECT_ROOT / "outputs" / "ligandmpnn" / "rfd3_hairpin_020826"

# Boltz output base
BOLTZ_BASE_DIR = PROJECT_ROOT / "outputs" / "boltz_predictions"
BOLTZ_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Ligand definition
LIGAND_SMILES_PATH = PROJECT_ROOT / "inputs" / "SMILES" / "DTZ.smiles"
LIGAND_SMILES = LIGAND_SMILES_PATH.read_text().strip()
LIGAND_ID = "DTZ"

# Batch settings
TOP_N_PER_DESIGN = 1           # Number of sequences to pre-filter per backbone
GLOBAL_TOP_PERCENT = 0.25      # Keep top 25% globally after per-design filtering
SORT_KEY = "overall_confidence"  # Primary key to sort LigandMPNN designs by
SORT_ASCENDING = False         # False for confidence-like metrics; True for loss-like metrics
SORT_KEY_CANDIDATES = [SORT_KEY, "overall_confidence", "global_score", "confidence_score", "score"]
DEDUPLICATE_SEQUENCES = True   # Remove duplicate amino-acid sequences before Boltz runs

if not (0 < GLOBAL_TOP_PERCENT <= 1):
    raise ValueError("GLOBAL_TOP_PERCENT must be in the interval (0, 1].")

# Boltz Runtime Settings (Optimized for Screening)
BOLTZ_SETTINGS = {
    "devices": 1,
    "accelerator": "gpu",
    "recycling_steps": 3,
    "sampling_steps": 200,      # Standard diffusion steps
    "diffusion_samples": 1,     # 1 sample per prompt for screening (vs 5 for high-qual)
    "step_scale": 1.638,
    "output_format": "pdb",
    "use_msa": False,           # De novo designs have no MSA
    "optimize_affinity": False, # Skip affinity optimization for initial screen (speed up)
}

# Slurm Settings
SLURM_PARTITION = "gpu"
SLURM_GPUS = 1
SLURM_GPU_TYPE = "A5500"          # Set to None to omit type constraint
SLURM_MEM = "12G"
SLURM_TIME = "0:20:00"
SLURM_ARRAY_MAX_CONCURRENT = 36

print(f"Project Root: {PROJECT_ROOT}")
print(f"Boltz Output: {BOLTZ_BASE_DIR}")
print(f"Ligand: {LIGAND_ID} ({LIGAND_SMILES})")

Project Root: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026
Boltz Output: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/boltz_predictions
Ligand: DTZ (O=C(N1C=C(C2=CC=CC=C2)[N-]3)C(CC4=CC=CC=C4)=NC1=C3C5=CC=CC=C5)


In [6]:
# =============================================================================
# 2) LOAD & SELECT DESIGNS from LigandMPNN
# =============================================================================
def get_latest_dir(base_dir: Path) -> Path:
    candidates = [p for p in base_dir.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No results found in {base_dir}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

try:
    MPNN_RESULTS_DIR = get_latest_dir(MPNN_BASE_DIR)
    print(f"Processing MPNN Batch: {MPNN_RESULTS_DIR.name}")
except FileNotFoundError as e:
    print(e)
    # create dummy for validation if needed
    MPNN_RESULTS_DIR = None

# Parse FASTA files
rows = []
if MPNN_RESULTS_DIR:
    search_pattern = "seqs/*.fa"
    found_fastas = list(MPNN_RESULTS_DIR.glob(search_pattern))
    
    print(f"Found {len(found_fastas)} FASTA files.")
    
    for fasta_path in tqdm(found_fastas, desc="Parsing FASTAs"):
        parent_design = fasta_path.stem.replace(".fa", "").replace(".fasta", "")
        
        for idx, record in enumerate(SeqIO.parse(str(fasta_path), "fasta")):
            # Skip template (idx 0) usually, check if it says 'template' or similar
            if idx == 0: continue 
            
            row = {
                "parent_design": parent_design,
                "name": f"{record.id}_seq_{idx}", # Ensure uniqueness by adding sequence index
                "sequence": str(record.seq),
                "source_file": str(fasta_path)
            }
            
            # Parse scores from description
            desc_parts = record.description.split()
            for part in desc_parts:
                if "=" in part:
                    k, v = part.split("=", 1)
                    try: row[k] = float(v)
                    except ValueError: row[k] = v
            rows.append(row)

    df_mpnn = pd.DataFrame(rows)
    print(f"Total sequences extracted: {len(df_mpnn)}")
    
    # Filter/Select Top N per design, then top global percentile
    if not df_mpnn.empty and SORT_KEY in df_mpnn.columns:
        selected_rows = []
        for name, group in df_mpnn.groupby("parent_design"):
            sorted_grp = group.sort_values(SORT_KEY, ascending=SORT_ASCENDING)
            selected_rows.append(sorted_grp.head(TOP_N_PER_DESIGN))
        
        df_selected = pd.concat(selected_rows, ignore_index=True)
        print(f"Selected Top {TOP_N_PER_DESIGN} per design: {len(df_selected)} candidates")

        if not df_selected.empty:
            n_keep = max(1, math.ceil(len(df_selected) * GLOBAL_TOP_PERCENT))
            df_selected = df_selected.sort_values(SORT_KEY, ascending=SORT_ASCENDING).head(n_keep)
            print(f"Keeping global top {GLOBAL_TOP_PERCENT:.0%} by '{SORT_KEY}': {len(df_selected)} candidates")
    else:
        df_selected = df_mpnn.head(10) # Fallback
        print(f"Warning: Sort key '{SORT_KEY}' not found or empty DF. Taking first 10 or empty.")
else:
    df_selected = pd.DataFrame()
    print("No input data found.")

if not df_selected.empty:
    display(df_selected.head())

Processing MPNN Batch: 20260212_132203
Found 20000 FASTA files.


Parsing FASTAs:   0%|          | 0/20000 [00:00<?, ?it/s]

Total sequences extracted: 160000
Selected Top 1 per design: 20000 candidates
Keeping global top 25% by 'overall_confidence': 5000 candidates


,parent_design,name,sequence,source_file,id,T,seed,overall_confidence,ligand_confidence,seq_rec
6589,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,MTDEEIAEFLKAFYEALDAGDAETAADLLFGAGCKEIRLDVPDAGL...,/private/groups/yehlab/wsobolew/02_projects/co...,"8,","0.1,","21076,","0.7316,","0.7032,",0.5714
5836,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,MTDEEIAEFLKAFYEALDAGDAETAADLLFGAGCKEIRFDVPDDPL...,/private/groups/yehlab/wsobolew/02_projects/co...,"2,","0.1,","49380,","0.7311,","0.7238,",0.5000
16527,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,MTDEEIAEFLKAFYEALDAGDAETAADLLFGAGCKEIRLNLPGGGV...,/private/groups/yehlab/wsobolew/02_projects/co...,"8,","0.1,","80461,","0.7275,","0.7806,",0.5714
1915,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,MTDEEIAEFLKAFYEALDAGDAETAADLLFGAGCKEIRINLPGGGV...,/private/groups/yehlab/wsobolew/02_projects/co...,"1,","0.1,","62139,","0.7194,","0.7780,",0.8571
18901,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,rfd3_hairpin_config_20260208_170431_DR3M41_hai...,MTDEEIAEFLKAFYEALDAGDAETAADLLFGAGCKEIRVDLPDAPP...,/private/groups/yehlab/wsobolew/02_projects/co...,"2,","0.1,","73081,","0.7134,","0.7053,",0.8333


In [7]:
# =============================================================================
# 3) GENERATE BOLTZ INPUTS (YAML)
# =============================================================================
def sanitize_name(name: str, max_len: int = 120) -> str:
    clean = re.sub(r"[^A-Za-z0-9_.-]", "_", str(name))
    clean = re.sub(r"_+", "_", clean).strip("_")
    return clean[:max_len] if len(clean) > max_len else clean

# Create a new batch directory
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BATCH_NAME = f"boltz_hairpin_{timestamp}"
BATCH_DIR = BOLTZ_BASE_DIR / BATCH_NAME

YAML_DIR = BATCH_DIR / "yaml_inputs"
LOGS_DIR = BATCH_DIR / "logs"
PREDS_DIR = BATCH_DIR / "predictions"

for d in [YAML_DIR, LOGS_DIR, PREDS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

manifest_path = BATCH_DIR / "manifest_inputs.txt"
mapping_csv_path = BATCH_DIR / "selected_sequences_for_boltz.csv"

print(f"Generating inputs in: {BATCH_DIR}")

yaml_files = []
manifest_lines = []
name_counts = {}

if df_selected.empty:
    print("No sequences selected! Check input paths and filtering settings.")
else:
    if "name" not in df_selected.columns or "sequence" not in df_selected.columns:
        raise ValueError("df_selected must contain at least 'name' and 'sequence' columns.")

    if "SORT_KEY_EFFECTIVE" not in globals():
        SORT_KEY_EFFECTIVE = SORT_KEY if SORT_KEY in df_selected.columns else None

    export_cols = [c for c in ["parent_design", "name", "sequence", "source_file", SORT_KEY_EFFECTIVE] if c and c in df_selected.columns]
    df_selected[export_cols].to_csv(mapping_csv_path, index=False)
    print(f"Saved selected-sequence mapping: {mapping_csv_path}")

    for idx, row in tqdm(df_selected.iterrows(), total=len(df_selected), desc="Generating YAMLs"):
        raw_name = row["name"]
        safe_name = sanitize_name(raw_name)
        name_counts[safe_name] = name_counts.get(safe_name, 0) + 1
        if name_counts[safe_name] > 1:
            safe_name = f"{safe_name}_{name_counts[safe_name]-1}"

        sequence = row["sequence"]

        data = {
            "sequences": [
                {
                    "protein": {
                        "id": ["A"],
                        "sequence": sequence,
                        "msa": "empty"
                    }
                },
                {
                    "ligand": {
                        "id": ["B"],
                        "name": LIGAND_ID,
                        "smiles": LIGAND_SMILES
                    }
                }
            ]
        }

        if BOLTZ_SETTINGS["optimize_affinity"]:
            data["properties"] = [{"affinity": {"binder": "B"}}]

        yaml_name = f"{safe_name}.yaml"
        yaml_path = YAML_DIR / yaml_name
        with open(yaml_path, "w") as f:
            yaml.dump(data, f, sort_keys=False)

        yaml_files.append(yaml_path)
        manifest_lines.append(str(yaml_path))

    with open(manifest_path, "w") as f:
        f.write("\n".join(manifest_lines))

    print(f"Generated {len(yaml_files)} YAML files.")
    print(f"Manifest saved to: {manifest_path}")

Generating inputs in: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/boltz_predictions/boltz_hairpin_20260216_164316
Saved selected-sequence mapping: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/boltz_predictions/boltz_hairpin_20260216_164316/selected_sequences_for_boltz.csv


Generating YAMLs:   0%|          | 0/5000 [00:00<?, ?it/s]

Generated 5000 YAML files.
Manifest saved to: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/boltz_predictions/boltz_hairpin_20260216_164316/manifest_inputs.txt


In [8]:
# =============================================================================
# 4) GENERATE SLURM SUBMISSION SCRIPT
# =============================================================================
# This script uses a Slurm Job Array to process the manifest file
# Task ID -> Line number in manifest -> YAML file

script_path = BATCH_DIR / "submit_array.sh"

if len(yaml_files) == 0:
    print("No YAML files were generated. Skipping Slurm script creation.")
    script_path = None
else:
    array_max = min(SLURM_ARRAY_MAX_CONCURRENT, len(yaml_files))
    if SLURM_GPU_TYPE:
        gres_line = f"#SBATCH --gres=gpu:{SLURM_GPU_TYPE}:{SLURM_GPUS}"
    else:
        gres_line = f"#SBATCH --gres=gpu:{SLURM_GPUS}"

    slurm_script_content = f"""#!/bin/bash
##SBATCH --job-name=boltz_screen
#SBATCH --output={LOGS_DIR}/%A_%a.out
#SBATCH --error={LOGS_DIR}/%A_%a.err
#SBATCH --time={SLURM_TIME}
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=3
#SBATCH --mem={SLURM_MEM}
#SBATCH --partition={SLURM_PARTITION}
{gres_line}
#SBATCH --array=1-{len(yaml_files)}%{array_max}

# Stop on error
set -euo pipefail

# Initialize environment
eval "$(micromamba shell hook --shell bash)"
micromamba activate boltz

# Get YAML file from manifest based on Task ID
MANIFEST="{manifest_path}"
YAML_FILE=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" "$MANIFEST")
if [[ -z "$YAML_FILE" ]]; then
    echo "No YAML entry for task ID $SLURM_ARRAY_TASK_ID" >&2
    exit 1
fi
BASENAME=$(basename "$YAML_FILE" .yaml)

# Define clean output directory for this design
OUT_DIR="{PREDS_DIR}/$BASENAME"
mkdir -p "$OUT_DIR"

echo "Running Boltz for $BASENAME"
echo "Input: $YAML_FILE"
echo "Output: $OUT_DIR"

# Set optimizations
export CUDA_LAUNCH_BLOCKING=0
export PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:512
# Optimize precision
python -c "import torch; torch.set_float32_matmul_precision('medium')"

# Run Boltz
srun boltz predict "$YAML_FILE" \\
    --out_dir "$OUT_DIR" \\
    --devices {BOLTZ_SETTINGS['devices']} \\
    --accelerator {BOLTZ_SETTINGS['accelerator']} \\
    --recycling_steps {BOLTZ_SETTINGS['recycling_steps']} \\
    --sampling_steps {BOLTZ_SETTINGS['sampling_steps']} \\
    --diffusion_samples {BOLTZ_SETTINGS['diffusion_samples']} \\
    --step_scale {BOLTZ_SETTINGS['step_scale']} \\
    --output_format {BOLTZ_SETTINGS['output_format']} \\
    --cache /private/groups/yehlab/wsobolew/.boltz_cache_shared

echo "Done."
"""

    with open(script_path, "w") as f:
        f.write(slurm_script_content)

    # Make executable
    os.chmod(script_path, 0o755)

    print(f"Created submission script: {script_path}")

Created submission script: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/boltz_predictions/boltz_hairpin_20260216_164316/submit_array.sh


In [9]:
if script_path is None:
    print("No submission script available (no YAML inputs generated).")
else:
    !sbatch {script_path}

Submitted batch job 29790591
